In [59]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [7]:
X,y=make_classification(n_samples=1000,n_features=2,random_state=42,n_informative=2,n_redundant=0)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [48]:
class PerceptronNumpy():
  def __init__(self,n_iterations=100,learning_rate=0.01):
    self.n_iterations=n_iterations
    self.learning_rate=learning_rate
    self.weights=None
    self.bias=None

  def train(self,X_train,y_train):
    n_samples,n_features=X_train.shape
    self.weights=np.zeros(n_features)
    self.bias=0;
    for _ in range(self.n_iterations):
      for idx,x_i in enumerate(X_train):
        pred=np.dot(self.weights.T,x_i)+self.bias
        output=1 if pred>0 else 0
        loss=y_train[idx]-output
        self.weights=self.weights-self.learning_rate*loss*x_i
        self.bias=self.bias-self.learning_rate*loss

  def predict(self,X_test,y_test):
    acc=0;
    for i in (range(X_test.shape[0])):
      x_i=X_test[i]
      output=np.dot(x_i,self.weights)+self.bias
      pred=1 if output>0 else 0
      if pred==y_test[i]:
        acc+=1
    return acc/len(y_test)

In [49]:
obj=PerceptronNumpy(100,0.01)

In [50]:
obj.train(X_train,y_train)

In [51]:
acc=obj.predict(X_test,y_test)

In [52]:
acc

0.115

In [60]:
X_train_t=torch.tensor(X_train,dtype=torch.float32)
X_test_t=torch.tensor(X_test,dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)


In [66]:
class PerceptronTorch(nn.Module):
  def __init__(self,input_dim):
    super(PerceptronTorch,self).__init__()
    self.linear=nn.Linear(input_dim,1)

  def forward(self,x:torch.tensor)->torch.tensor:
    x=torch.sigmoid(self.linear(x))
    return x

model=PerceptronTorch(X_train.shape[1])
criterion=nn.BCELoss()
optimizer=optim.SGD(model.parameters(),lr=0.1)
epochs=100
for i in range(epochs):
  optimizer.zero_grad()
  outputs=model.forward(X_train_t)
  loss=criterion(outputs,y_train_t)
  loss.backward()
  optimizer.step()

with torch.no_grad():
  predictions=(model(X_test_t)>=0.5).float()
  accuracy=(predictions==y_test_t).float().mean();
  print(accuracy)








tensor(0.8800)
